# GrAId — Remote OCR Server (Surya + Qwen2.5-VL, Colab GPU)

Serves the `/health` and `/ocr` endpoints that `ocr_pipeline.py`'s `_run_remote_ocr_pipeline()` expects, so the FastAPI backend can delegate handwriting OCR to a Colab GPU instead of a local one.

**How to use**
1. `Runtime -> Change runtime type -> T4 GPU`, then `Runtime -> Run all`.
2. When prompted, paste your ngrok authtoken (free account at https://dashboard.ngrok.com/get-started/your-authtoken).
3. Copy the public URL printed by the last cell.
4. On the machine running the FastAPI backend, set `REMOTE_OCR_URL=<that url>` (env var or `.env`) and start/restart the backend.

**Known constraint (see thesis Limitations):** free-tier Colab sessions disconnect after periods of inactivity or after ~12h, and GPU availability varies. If a session dies, re-run this notebook and update `REMOTE_OCR_URL` with the new ngrok URL.

## 1. Install dependencies
Versions pinned to match `requirements.txt` in the main GrAId repo so transcription behavior is identical to what the thesis describes.

In [ ]:
try:
    import surya
    import transformers
    import accelerate
    import bitsandbytes
    import qwen_vl_utils
    import fastapi
    import uvicorn
    import pyngrok
    import nest_asyncio
    import cv2

    print("Required packages already importable in this runtime — skipping install.")
    print("(This only triggers after Runtime > Restart session, which keeps the same VM. ")
    print("A full disconnect/reconnect gets a fresh VM and will always need to reinstall.)")
except ImportError:
    !pip install -q "surya-ocr==0.13.1" "transformers>=4.47.0" "accelerate>=1.0.0" \
        "bitsandbytes>=0.44.1" "qwen-vl-utils>=0.0.8" torchvision \
        "fastapi>=0.115.0" "uvicorn[standard]>=0.30.0" "python-multipart>=0.0.9" \
        pyngrok nest_asyncio opencv-python-headless "pillow>=10.2.0,<11.0.0"

## 2. Mount Google Drive for a persistent model cache
Colab's local disk is wiped every time the runtime resets, so without this every re-run re-downloads Qwen2.5-VL-7B's full-precision weights (~15GB) from Hugging Face — several minutes to tens of minutes depending on Colab's network. Pointing `HF_HOME` at a folder in Drive means the first run downloads once; later sessions reuse the cached files.

**Before running this cell:** check you have ~20GB free in this Google account's Drive (`drive.google.com` storage meter, bottom-left). Free Drive accounts get 15GB total shared with Gmail/Photos — if that's not enough room, skip this cell and re-download each session instead (slower, but requires no Drive space).

In [ ]:
import os

from google.colab import drive

drive.mount("/content/drive")

CACHE_DIR = "/content/drive/MyDrive/GrAId_model_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

# Must be set before transformers/huggingface_hub are imported anywhere below —
# huggingface_hub reads these at import time, not lazily.
os.environ["HF_HOME"] = CACHE_DIR
os.environ["HF_HUB_CACHE"] = os.path.join(CACHE_DIR, "hub")

print(f"Model cache directory: {CACHE_DIR}")
print("First run downloads ~15GB into Drive and takes a while; later sessions reuse it and skip the download.")

## 3. Imports + config

In [ ]:
import base64
import io
from dataclasses import dataclass
from typing import Any, Optional

import cv2
import numpy as np
import torch
from PIL import Image, ImageDraw
from qwen_vl_utils import process_vision_info
from transformers import AutoModelForVision2Seq, AutoProcessor, BitsAndBytesConfig
from surya.detection import DetectionPredictor

QWEN_MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "No GPU detected. Go to Runtime -> Change runtime type -> Hardware accelerator -> GPU (T4), then Run all again."
    )

## 4. Load Surya (line detection) + Qwen2.5-VL-7B-Instruct (4-bit)
Same loading code as the local branch of `ocr_pipeline.load_models()`, so results match what the thesis validates against.

In [ ]:
@dataclass
class Models:
    surya_detector: Any
    qwen_model: Any
    qwen_processor: Any


MODELS: Optional[Models] = None


def load_models() -> None:
    global MODELS
    if MODELS is not None:
        print("[Models] Already loaded.")
        return

    print("[Models] Loading Surya detection model...")
    surya_detector = DetectionPredictor()
    try:
        surya_detector.to("cuda")
        print("[Models] Surya loaded on CUDA.")
    except Exception:
        print("[Models] Surya CUDA move failed — running on CPU.")

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )

    print(f"[Models] Loading Qwen processor ({QWEN_MODEL_ID})...")
    qwen_processor = AutoProcessor.from_pretrained(QWEN_MODEL_ID, trust_remote_code=True)

    print("[Models] Loading Qwen model in 4-bit (this may take a few minutes)...")
    qwen_model = AutoModelForVision2Seq.from_pretrained(
        QWEN_MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.float16,
        trust_remote_code=True,
    )
    qwen_model.eval()
    print("[Models] Qwen loaded and ready.")

    MODELS = Models(
        surya_detector=surya_detector,
        qwen_model=qwen_model,
        qwen_processor=qwen_processor,
    )
    print("[Models] All models ready.")


load_models()

## 5. OCR pipeline helpers
Copied from `ocr_pipeline.py`'s local-inference path so the remote server produces the same `(text, boxes, boxed_image)` shape the backend already expects.

In [ ]:
def _pil_to_cv2_bgr(img: Image.Image) -> np.ndarray:
    rgb = np.array(img.convert("RGB"))
    return cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)


def preprocess_for_detection(img: Image.Image) -> Image.Image:
    """Grayscale → denoise → adaptive threshold → PIL RGB (Surya expects PIL)."""
    bgr = _pil_to_cv2_bgr(img)
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    den = cv2.fastNlMeansDenoising(gray, None, h=12, templateWindowSize=7, searchWindowSize=21)
    thr = cv2.adaptiveThreshold(
        den, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 31, 15
    )
    rgb = cv2.cvtColor(thr, cv2.COLOR_GRAY2RGB)
    return Image.fromarray(rgb)


def draw_boxes(image: Image.Image, boxes_xyxy: list) -> Image.Image:
    out = image.convert("RGB").copy()
    draw = ImageDraw.Draw(out)
    for (x1, y1, x2, y2) in boxes_xyxy:
        draw.rectangle([x1, y1, x2, y2], outline=(255, 0, 0), width=2)
    return out


def encode_png_base64(img: Image.Image) -> str:
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode("utf-8")


def _extract_surya_xyxy(pred: Any) -> list:
    boxes: list = []
    if pred is None:
        return boxes
    bboxes = getattr(pred, "bboxes", None)
    if not bboxes:
        return boxes
    for b in bboxes:
        rect = getattr(b, "bbox", None)
        if not rect or len(rect) != 4:
            continue
        x1, y1, x2, y2 = rect
        boxes.append((int(x1), int(y1), int(x2), int(y2)))
    return boxes


def crop_lines(original_rgb: Image.Image, boxes_xyxy: list) -> list:
    w, h = original_rgb.size
    crops: list = []
    for (x1, y1, x2, y2) in boxes_xyxy:
        x1c, y1c = max(0, x1), max(0, y1)
        x2c, y2c = min(w, x2), min(h, y2)
        if x2c <= x1c or y2c <= y1c:
            continue
        crops.append(original_rgb.crop((x1c, y1c, x2c, y2c)))
    return crops


def qwen_ocr_lines(line_images: list) -> list:
    assert MODELS is not None
    prompt = "Transcribe the handwritten text in this image line. Output only the text."
    texts: list = []
    for img in line_images:
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": img},
                    {"type": "text", "text": prompt},
                ],
            }
        ]
        text = MODELS.qwen_processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = MODELS.qwen_processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            return_tensors="pt",
        )
        inputs = {k: v.to(MODELS.qwen_model.device) for k, v in inputs.items() if hasattr(v, "to")}

        with torch.inference_mode():
            output_ids = MODELS.qwen_model.generate(
                **inputs,
                max_new_tokens=128,
                do_sample=False,
            )

        prompt_len = inputs["input_ids"].shape[1]
        gen_only = output_ids[:, prompt_len:]
        decoded = MODELS.qwen_processor.batch_decode(gen_only, skip_special_tokens=True)
        texts.append(decoded[0].strip())
    return texts


def run_ocr_pipeline(original: Image.Image):
    """Mirrors ocr_pipeline.run_ocr_pipeline's local (non-remote) branch.

    Returns (full_text, boxes_sorted, boxed_image).
    """
    assert MODELS is not None

    det_img = preprocess_for_detection(original)
    preds = MODELS.surya_detector([det_img])
    pred0 = preds[0] if preds else None
    boxes = _extract_surya_xyxy(pred0)
    boxes_sorted = sorted(boxes, key=lambda b: (b[1], b[0]))

    boxed = draw_boxes(original, boxes_sorted)

    if not boxes_sorted:
        return "", [], boxed

    line_crops = crop_lines(original, boxes_sorted)
    line_texts = qwen_ocr_lines(line_crops)
    clean_lines = [" ".join(t.split()) for t in line_texts if t and t.strip()]
    full_text = "\n".join(clean_lines).strip()

    return full_text, boxes_sorted, boxed


print("OCR pipeline helpers defined.")

## 6. FastAPI app — `/health` and `/ocr`
Contract matches `_run_remote_ocr_pipeline()` in `ocr_pipeline.py`:
- `GET /health` → 200 OK once models are loaded.
- `POST /ocr` (multipart `file`) → `{"text": str, "boxes": [[x1,y1,x2,y2], ...], "boxed_image_png_base64": str}`.

In [ ]:
from fastapi import FastAPI, File, HTTPException, UploadFile
from fastapi.responses import JSONResponse

app = FastAPI(title="GrAId Remote OCR Server")


@app.get("/health")
def health():
    if MODELS is None:
        raise HTTPException(status_code=503, detail="Models not loaded")
    return {"status": "ok"}


@app.post("/ocr")
async def ocr(file: UploadFile = File(...)):
    if MODELS is None:
        raise HTTPException(status_code=503, detail="Models not loaded")

    raw = await file.read()
    try:
        original = Image.open(io.BytesIO(raw)).convert("RGB")
    except Exception as e:
        raise HTTPException(status_code=400, detail=f"Invalid image: {e}")

    full_text, boxes, boxed = run_ocr_pipeline(original)

    return JSONResponse(
        {
            "text": full_text,
            "boxes": [list(b) for b in boxes],
            "boxed_image_png_base64": encode_png_base64(boxed),
        }
    )


print("FastAPI app defined.")

## 7. Run the server + open an ngrok tunnel
Paste your ngrok authtoken below (kept only in this Colab runtime, never written to the repo).

In [ ]:
from getpass import getpass

NGROK_AUTH_TOKEN = getpass("Paste your ngrok authtoken (hidden input): ").strip()

In [ ]:
import threading
import time

import nest_asyncio
import requests
import uvicorn
from pyngrok import ngrok

nest_asyncio.apply()

PORT = 8000

# Authenticate BEFORE any other pyngrok call - pyngrok lazily starts the local
# ngrok agent process on first use, and a modern ngrok agent refuses to start
# at all without a token (ERR_NGROK_4018), even just to list/close tunnels.
if not NGROK_AUTH_TOKEN:
    raise RuntimeError("NGROK_AUTH_TOKEN is empty - re-run the cell above and paste your authtoken.")
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Close any tunnels left over from a previous run of this cell in this session
try:
    for t in ngrok.get_tunnels():
        ngrok.disconnect(t.public_url)
except Exception:
    pass

# If this cell already ran once in this session (e.g. you're re-running it
# because the ngrok tunnel dropped, without restarting the kernel), stop the
# old server first - otherwise the new one fails with "address already in
# use" while the old one silently keeps serving underneath it.
if "_graid_server" in globals() and _graid_server is not None:
    print("Stopping previous server instance...")
    _graid_server.should_exit = True
    time.sleep(2)

_graid_server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="info"))


def _run_server():
    _graid_server.run()


thread = threading.Thread(target=_run_server, daemon=True)
thread.start()
time.sleep(3)

resp = requests.get(f"http://127.0.0.1:{PORT}/health", timeout=10)
print("Local health check:", resp.status_code, resp.json())

public_url = ngrok.connect(PORT, "http").public_url
print("\n=== Remote OCR server is live ===")
print(f"REMOTE_OCR_URL={public_url}")
print("\nSet this as the REMOTE_OCR_URL env var wherever the GrAId FastAPI backend runs, then (re)start it.")

## 8. Keep-alive
Run this cell last and leave it running while grading a batch — it blocks the notebook (by design) so the Colab runtime doesn't idle-disconnect mid-session. Interrupt the cell (■) when you're done; the server + tunnel stay up until the runtime itself is reset.

In [ ]:
print("Server running. Interrupt this cell (Runtime > Interrupt execution) when you're done grading.")
try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print("Interrupted — server thread keeps running in the background until the runtime resets.")